In [ ]:
from google.colab import files

# Upload from your PC (you can select multiple files at once)
uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded.keys():
    print(" -", name)


Saving CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf to CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf
Saving image (1).png to image (1).png
Saving image.png to image.png
Saving CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg to CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg
Saving CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf to CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf

Uploaded files:
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf
 - image (1).png
 - image.png
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf


In [ ]:
import os, glob

print("Current folder:", os.getcwd())
print("\nAll uploaded files in this session:")
for f in sorted(glob.glob("*")):
    print(" -", f)


Current folder: /content

All uploaded files in this session:
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf
 - image (1).png
 - image.png
 - sample_data


In [ ]:
!pip -q install pymupdf opencv-python networkx pandas numpy matplotlib pydantic gradio pymoo ezdxf shapely


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 42.0 MB/s eta 0:00:00


In [ ]:
import glob, os
from pathlib import Path

patterns = ["*.pdf","*.png","*.jpg","*.jpeg","*.dxf","*.csv","*.xlsx","*.dwg"]
found = sorted(set([f for p in patterns for f in glob.glob(p)]))

print("Found in Colab working directory:")
for f in found:
    print(" -", f)

PDF_PATH = next((f for f in found if f.lower().endswith(".pdf") and ("p&id" in f.lower() or "pid" in f.lower())), None)
if PDF_PATH is None:
    PDF_PATH = next((f for f in found if f.lower().endswith(".pdf")), None)

IMG_PATHS = [f for f in found if f.lower().endswith((".png",".jpg",".jpeg"))]
DXF_PATH  = next((f for f in found if f.lower().endswith(".dxf")), None)
DWG_PATH  = next((f for f in found if f.lower().endswith(".dwg")), None)

print("\nAuto-selected:")
print("PDF_PATH:", PDF_PATH)
print("IMG_PATHS:", IMG_PATHS)
print("DXF_PATH:", DXF_PATH)
print("DWG_PATH:", DWG_PATH)

if DWG_PATH and not DXF_PATH:
    print("\nNOTE: DWG can't be reliably parsed in Colab. Convert DWG → DXF on PC and upload DXF for geometry extraction.")


Found in Colab working directory:
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf
 - CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf
 - image (1).png
 - image.png

Auto-selected:
PDF_PATH: CR11210003_P&ID OF PYROLYSIS PILOT PLANT.pdf
IMG_PATHS: ['image (1).png', 'image.png']
DXF_PATH: CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dxf
DWG_PATH: CR11210003_P&ID OF PYROLYSIS PILOT PLANT.dwg


In [ ]:
import numpy as np
import cv2

pid_image = None

if PDF_PATH:
    import fitz  # PyMuPDF
    doc = fitz.open(PDF_PATH)
    page = doc[0]  # change page index if needed
    pix = page.get_pixmap(dpi=300)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    pid_image = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    print("Loaded image from PDF:", pid_image.shape)

elif IMG_PATHS:
    pid_image = cv2.imread(IMG_PATHS[0])
    print("Loaded image from PNG/JPG:", IMG_PATHS[0], pid_image.shape)

else:
    raise FileNotFoundError("Upload at least one P&ID PDF or PNG.")

cv2.imwrite("pid_page.png", pid_image)
print("Saved working image: pid_page.png")


Loaded image from PDF: (3509, 4963, 3)
Saved working image: pid_page.png


In [ ]:
import json
from dataclasses import dataclass, asdict

@dataclass
class Node:
    id: str
    kind: str
    tag: str
    x: int = 0
    y: int = 0

@dataclass
class Edge:
    id: str
    kind: str
    src: str
    dst: str
    tag: str = ""

pid = {
    "nodes": [],
    "edges": [],
    "meta": {
        "uploaded_files": {
            "pdf": PDF_PATH,
            "images": IMG_PATHS,
            "dxf": DXF_PATH,
            "dwg": DWG_PATH
        },
        "note": "Edit nodes/edges to match exact tags from your P&ID."
    }
}

def add_node(id, kind, tag, x=0, y=0):
    pid["nodes"].append(asdict(Node(id=id, kind=kind, tag=tag, x=x, y=y)))

def add_edge(id, kind, src, dst, tag=""):
    pid["edges"].append(asdict(Edge(id=id, kind=kind, src=src, dst=dst, tag=tag)))

# ======= STARTER fast pyrolysis network (edit tag names to match your drawing) =======
add_node("N_FEED", "feeder", "Biomass_Feeding")
add_node("N_REAC", "reactor", "Pyrolysis_Reactor")
add_node("N_CYC",  "cyclone", "Cyclone")
add_node("N_CHAR", "solid_collector", "Char_Collector")
add_node("N_COND", "condenser", "Condenser")
add_node("N_CHIL", "chiller", "Chiller")
add_node("N_BLOW", "blower", "Suction_Blower")
add_node("N_FLAR", "flare", "Flaring_Unit")
add_node("N_N2",   "utility", "N2_Supply")

add_edge("E1","solid","N_FEED","N_REAC","Feed_to_Reactor")
add_edge("E2","vapour","N_REAC","N_CYC","Vapour_to_Cyclone")
add_edge("E3","solid","N_CYC","N_CHAR","Char_to_Collector")
add_edge("E4","vapour","N_CYC","N_COND","Vapour_to_Condenser")
add_edge("E5","liquid","N_COND","N_CHIL","BioOil_to_Chiller")
add_edge("E6","gas","N_CHIL","N_BLOW","Noncondensable_to_Blower")
add_edge("E7","gas","N_BLOW","N_FLAR","To_Flare")
add_edge("E8","N2","N_N2","N_REAC","N2_to_Reactor")

with open("digital_pid.json","w") as f:
    json.dump(pid,f,indent=2)

print("Saved digital_pid.json")
print("Nodes:", len(pid["nodes"]), "Edges:", len(pid["edges"]))


Saved digital_pid.json
Nodes: 9 Edges: 8


In [ ]:
import networkx as nx, json

with open("digital_pid.json","r") as f:
    pid = json.load(f)

G = nx.DiGraph()
for n in pid["nodes"]:
    G.add_node(n["id"], **n)
for e in pid["edges"]:
    G.add_edge(e["src"], e["dst"], **e)

print("Graph ready:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("Nodes sample:", list(G.nodes(data=True))[:3])
print("Edges sample:", list(G.edges(data=True))[:3])


Graph ready: 9 nodes, 8 edges
Nodes sample: [('N_FEED', {'id': 'N_FEED', 'kind': 'feeder', 'tag': 'Biomass_Feeding', 'x': 0, 'y': 0}), ('N_REAC', {'id': 'N_REAC', 'kind': 'reactor', 'tag': 'Pyrolysis_Reactor', 'x': 0, 'y': 0}), ('N_CYC', {'id': 'N_CYC', 'kind': 'cyclone', 'tag': 'Cyclone', 'x': 0, 'y': 0})]
Edges sample: [('N_FEED', 'N_REAC', {'id': 'E1', 'kind': 'solid', 'src': 'N_FEED', 'dst': 'N_REAC', 'tag': 'Feed_to_Reactor'}), ('N_REAC', 'N_CYC', {'id': 'E2', 'kind': 'vapour', 'src': 'N_REAC', 'dst': 'N_CYC', 'tag': 'Vapour_to_Cyclone'}), ('N_CYC', 'N_CHAR', {'id': 'E3', 'kind': 'solid', 'src': 'N_CYC', 'dst': 'N_CHAR', 'tag': 'Char_to_Collector'})]


In [ ]:
import numpy as np

def pyrolysis_yields(u):
    """
    PoC correlations (replace with pilot-data ML later).
    Returns kg/h: bio_oil, char, gas, flare
    """
    F = float(u["feed_kgph"])
    T = float(u["reactor_C"])
    moisture = float(u["moisture_wtfrac"])    # 0-1
    tau = float(u["vap_res_time_s"])
    Tc = float(u["condenser_C"])
    eta_cyc = float(u["cyclone_eff"])
    gas_use = float(u["gas_use_kgph"])

    # Oil yield: increases with T to optimum then drops (secondary cracking)
    oil_y = 0.55 + 0.0015*(T-450) - 0.000010*(T-500)**2
    oil_y -= 0.25*moisture
    oil_y -= 0.02*max(0.0, tau-2.0)
    oil_y = float(np.clip(oil_y, 0.20, 0.75))

    # Char yield: decreases with T; moisture increases char
    char_y = 0.18 - 0.0006*(T-450) + 0.10*moisture
    char_y = float(np.clip(char_y, 0.05, 0.35))

    # Condenser capture efficiency
    cond_eff = float(np.clip(0.65 + 0.01*(40 - Tc), 0.2, 0.98))

    bio_oil = F * oil_y * cond_eff
    char = F * char_y * eta_cyc
    gas = max(0.0, F - bio_oil - char)
    flare = max(0.0, gas - gas_use)

    return {"bio_oil_kgph": bio_oil, "char_kgph": char, "gas_kgph": gas, "flare_kgph": flare}

def profit(u, prices):
    y = pyrolysis_yields(u)
    revenue = y["bio_oil_kgph"]*prices["bio_oil_rs_per_kg"] + y["char_kgph"]*prices["char_rs_per_kg"]
    power_cost = u["heater_kW"] * prices["elec_rs_per_kWh"]
    flare_penalty = y["flare_kgph"] * prices["flare_penalty_rs_per_kg"]
    return revenue - power_cost - flare_penalty, y

# Example base case (edit values any time)
u0 = dict(
    feed_kgph=15,
    reactor_C=450,
    moisture_wtfrac=0.10,
    vap_res_time_s=1.5,
    condenser_C=40,
    cyclone_eff=0.95,
    gas_use_kgph=1.0,
    heater_kW=200
)

prices = dict(
    bio_oil_rs_per_kg=35,
    char_rs_per_kg=8,
    elec_rs_per_kWh=10,
    flare_penalty_rs_per_kg=5
)

p0, y0 = profit(u0, prices)
print("Outputs:", y0)
print("Profit (Rs/h):", p0)


Outputs: {'bio_oil_kgph': 4.875, 'char_kgph': 2.7075, 'gas_kgph': 7.4175, 'flare_kgph': 6.4175}
Profit (Rs/h): -1839.8025


In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.termination import get_termination
import numpy as np

# decision variables: reactor_C, residence time, condenser_C, gas_use
xl = np.array([430, 0.8, 20, 0.0])
xu = np.array([550, 3.0, 60, 5.0])

class PyroProblem(Problem):
    def __init__(self, base_u):
        super().__init__(n_var=4, n_obj=3, n_constr=0, xl=xl, xu=xu)
        self.base_u = base_u

    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for row in X:
            u = dict(self.base_u)
            u["reactor_C"] = float(row[0])
            u["vap_res_time_s"] = float(row[1])
            u["condenser_C"] = float(row[2])
            u["gas_use_kgph"] = float(row[3])

            y = pyrolysis_yields(u)
            F.append([-y["bio_oil_kgph"], y["char_kgph"], y["flare_kgph"]])
        out["F"] = np.array(F)

res = minimize(PyroProblem(u0), NSGA2(pop_size=60), termination=get_termination("n_gen", 80),
               seed=1, verbose=False)

F = res.F
X = res.X

# pick a practical point: oil priority + flare reduction
best = np.argmin(F[:,0] + 2*F[:,2])
xbest = X[best]

u_best = dict(u0)
u_best["reactor_C"] = float(xbest[0])
u_best["vap_res_time_s"] = float(xbest[1])
u_best["condenser_C"] = float(xbest[2])
u_best["gas_use_kgph"] = float(xbest[3])

print("Best settings:", u_best)
print("Best outputs:", pyrolysis_yields(u_best))



Best settings: {'feed_kgph': 15, 'reactor_C': 550.0, 'moisture_wtfrac': 0.1, 'vap_res_time_s': 1.819655461271514, 'condenser_C': 20.0, 'cyclone_eff': 0.95, 'gas_use_kgph': 4.980503820187294, 'heater_kW': 200}
Best outputs: {'bio_oil_kgph': 8.287500000000001, 'char_kgph': 1.8525, 'gas_kgph': 4.8599999999999985, 'flare_kgph': 0.0}


In [ ]:
import gradio as gr

def run_dashboard(feed_kgph, reactor_C, moisture_pct, tau_s, condenser_C, cyclone_eff, gas_use_kgph, heater_kW,
                  oil_price, char_price, elec_price, flare_penalty):
    u = dict(
        feed_kgph=feed_kgph,
        reactor_C=reactor_C,
        moisture_wtfrac=moisture_pct/100,
        vap_res_time_s=tau_s,
        condenser_C=condenser_C,
        cyclone_eff=cyclone_eff,
        gas_use_kgph=gas_use_kgph,
        heater_kW=heater_kW
    )
    prices = dict(
        bio_oil_rs_per_kg=oil_price,
        char_rs_per_kg=char_price,
        elec_rs_per_kWh=elec_price,
        flare_penalty_rs_per_kg=flare_penalty
    )
    prof, y = profit(u, prices)
    return {**y, "profit_rs_per_h": prof}

demo = gr.Interface(
    fn=run_dashboard,
    inputs=[
        gr.Slider(1, 200, value=15, step=1, label="Feed rate (kg/h)"),
        gr.Slider(350, 650, value=450, step=5, label="Reactor Temperature (°C)"),
        gr.Slider(0, 30, value=10, step=1, label="Moisture (%)"),
        gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="Vapour residence time (s)"),
        gr.Slider(10, 80, value=40, step=1, label="Condenser temperature (°C)"),
        gr.Slider(0.70, 0.99, value=0.95, step=0.01, label="Cyclone efficiency"),
        gr.Slider(0, 20, value=1.0, step=0.1, label="Gas used internally (kg/h)"),
        gr.Slider(0, 500, value=200, step=10, label="Heater power (kW)"),
        gr.Slider(0, 200, value=35, step=1, label="Bio-oil price (Rs/kg)"),
        gr.Slider(0, 50, value=8, step=1, label="Char price (Rs/kg)"),
        gr.Slider(0, 30, value=10, step=1, label="Electricity (Rs/kWh)"),
        gr.Slider(0, 50, value=5, step=1, label="Flare penalty (Rs/kg)")
    ],
    outputs=gr.JSON(label="Outputs"),
    title="Fast Pyrolysis Digital Twin (PoC)"
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94a18f46c7c13cdfc2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import ezdxf, json

if DXF_PATH is None:
    print("No DXF uploaded. Convert DWG → DXF on PC and upload DXF to extract geometry.")
else:
    doc = ezdxf.readfile(DXF_PATH)
    msp = doc.modelspace()

    text_items = []
    for e in msp:
        t = e.dxftype()
        if t == "TEXT":
            ins = e.dxf.insert
            text_items.append({"type":"TEXT","text":e.dxf.text, "x":float(ins[0]), "y":float(ins[1])})
        elif t == "MTEXT":
            ins = e.dxf.insert
            text_items.append({"type":"MTEXT","text":e.text, "x":float(ins[0]), "y":float(ins[1])})

    with open("dxf_text_items.json","w") as f:
        json.dump(text_items, f, indent=2)

    print("Saved dxf_text_items.json with", len(text_items), "text elements (use to auto-suggest tags).")


Saved dxf_text_items.json with 1107 text elements (use to auto-suggest tags).


In [ ]:
# Sanity test: outputs MUST change with reactor temperature
uA = dict(feed_kgph=15, reactor_C=430, moisture_wtfrac=0.10, vap_res_time_s=1.5,
          condenser_C=40, cyclone_eff=0.95, gas_use_kgph=1.0, heater_kW=200)
uB = dict(uA); uB["reactor_C"] = 520

print("At 430C:", pyrolysis_yields(uA))
print("At 520C:", pyrolysis_yields(uB))


At 430C: {'bio_oil_kgph': 4.3485000000000005, 'char_kgph': 2.8785000000000003, 'gas_kgph': 7.772999999999998, 'flare_kgph': 6.772999999999998}
At 520C: {'bio_oil_kgph': 6.1035, 'char_kgph': 2.109, 'gas_kgph': 6.7875, 'flare_kgph': 5.7875}


In [ ]:
# If you didn't run the earlier function cells, run this whole cell.

import numpy as np

def pyrolysis_yields(u):
    F = float(u["feed_kgph"])
    T = float(u["reactor_C"])
    moisture = float(u["moisture_wtfrac"])
    tau = float(u["vap_res_time_s"])
    Tc = float(u["condenser_C"])
    eta_cyc = float(u["cyclone_eff"])
    gas_use = float(u["gas_use_kgph"])

    # Stronger sensitivity so changes are visible
    oil_y = 0.52 + 0.0025*(T-450) - 0.000015*(T-500)**2
    oil_y -= 0.30*moisture
    oil_y -= 0.03*max(0.0, tau-1.2)
    oil_y = float(np.clip(oil_y, 0.10, 0.80))

    char_y = 0.20 - 0.0008*(T-450) + 0.12*moisture
    char_y = float(np.clip(char_y, 0.05, 0.40))

    cond_eff = float(np.clip(0.60 + 0.015*(35 - Tc), 0.2, 0.98))

    bio_oil = F * oil_y * cond_eff
    char = F * char_y * eta_cyc
    gas = max(0.0, F - bio_oil - char)
    flare = max(0.0, gas - gas_use)

    return {"bio_oil_kgph": bio_oil, "char_kgph": char, "gas_kgph": gas, "flare_kgph": flare}

def profit(u, prices):
    y = pyrolysis_yields(u)
    revenue = y["bio_oil_kgph"]*prices["bio_oil_rs_per_kg"] + y["char_kgph"]*prices["char_rs_per_kg"]
    power_cost = float(u["heater_kW"]) * prices["elec_rs_per_kWh"]
    flare_penalty = y["flare_kgph"] * prices["flare_penalty_rs_per_kg"]
    return revenue - power_cost - flare_penalty, y

uA = dict(feed_kgph=15, reactor_C=430, moisture_wtfrac=0.10, vap_res_time_s=1.5,
          condenser_C=40, cyclone_eff=0.95, gas_use_kgph=1.0, heater_kW=200)
uB = dict(uA); uB["reactor_C"] = 520

print("430C:", pyrolysis_yields(uA))
print("520C:", pyrolysis_yields(uB))


430C: {'bio_oil_kgph': 2.8153125000000006, 'char_kgph': 3.249, 'gas_kgph': 8.935687499999998, 'flare_kgph': 7.9356874999999985}
520C: {'bio_oil_kgph': 5.11875, 'char_kgph': 2.2230000000000003, 'gas_kgph': 7.658249999999999, 'flare_kgph': 6.658249999999999}


In [ ]:
!pip -q install gradio

import gradio as gr

def run_dashboard(feed_kgph, reactor_C, moisture_pct, tau_s, condenser_C, cyclone_eff, gas_use_kgph, heater_kW,
                  oil_price, char_price, elec_price, flare_penalty):

    u = dict(
        feed_kgph=float(feed_kgph),
        reactor_C=float(reactor_C),
        moisture_wtfrac=float(moisture_pct)/100.0,
        vap_res_time_s=float(tau_s),
        condenser_C=float(condenser_C),
        cyclone_eff=float(cyclone_eff),
        gas_use_kgph=float(gas_use_kgph),
        heater_kW=float(heater_kW)
    )
    prices = dict(
        bio_oil_rs_per_kg=float(oil_price),
        char_rs_per_kg=float(char_price),
        elec_rs_per_kWh=float(elec_price),
        flare_penalty_rs_per_kg=float(flare_penalty)
    )

    prof, y = profit(u, prices)
    y["profit_rs_per_h"] = prof

    # round so changes are easy to see
    return {k: round(float(v), 4) for k, v in y.items()}

with gr.Blocks() as demo:
    gr.Markdown("## Fast Pyrolysis Digital Twin (LIVE updates)")

    with gr.Row():
        feed_kgph = gr.Slider(1, 200, value=15, step=1, label="Feed (kg/h)")
        reactor_C = gr.Slider(350, 650, value=450, step=5, label="Reactor T (°C)")
        moisture_pct = gr.Slider(0, 30, value=10, step=1, label="Moisture (%)")
        tau_s = gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="Residence time (s)")

    with gr.Row():
        condenser_C = gr.Slider(10, 80, value=40, step=1, label="Condenser T (°C)")
        cyclone_eff = gr.Slider(0.70, 0.99, value=0.95, step=0.01, label="Cyclone eff")
        gas_use_kgph = gr.Slider(0, 20, value=1.0, step=0.1, label="Gas used internally (kg/h)")
        heater_kW = gr.Slider(0, 500, value=200, step=10, label="Heater (kW)")

    with gr.Row():
        oil_price = gr.Slider(0, 200, value=35, step=1, label="Bio-oil price (Rs/kg)")
        char_price = gr.Slider(0, 50, value=8, step=1, label="Char price (Rs/kg)")
        elec_price = gr.Slider(0, 30, value=10, step=1, label="Electricity (Rs/kWh)")
        flare_penalty = gr.Slider(0, 50, value=5, step=1, label="Flare penalty (Rs/kg)")

    out = gr.JSON(label="Outputs")

    inputs = [feed_kgph, reactor_C, moisture_pct, tau_s, condenser_C, cyclone_eff, gas_use_kgph, heater_kW,
              oil_price, char_price, elec_price, flare_penalty]

    # Update on any slider movement
    for c in inputs:
        c.change(run_dashboard, inputs=inputs, outputs=out)

    # Initial compute at load
    demo.load(run_dashboard, inputs=inputs, outputs=out)

demo.launch(share=True, debug=True)



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://204e88bbf6b2dc1c8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# FAST scenario: fewer frames
df = simulate_scenario(minutes=2, dt_s=2)   # 2 minutes, step 2s => ~61 frames

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(7,4))
ax.set_xlim(df["t_s"].min(), df["t_s"].max())
ax.set_ylim(0, max(df["bio_oil_kgph"].max(), df["char_kgph"].max(), df["flare_kgph"].max()) * 1.2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("kg/h")

line_oil, = ax.plot([], [], label="Bio-oil")
line_char, = ax.plot([], [], label="Char")
line_flare, = ax.plot([], [], label="Flare")
ax.legend()

def init():
    line_oil.set_data([], [])
    line_char.set_data([], [])
    line_flare.set_data([], [])
    return line_oil, line_char, line_flare

def update(i):
    x = df["t_s"].iloc[:i+1]
    line_oil.set_data(x, df["bio_oil_kgph"].iloc[:i+1])
    line_char.set_data(x, df["char_kgph"].iloc[:i+1])
    line_flare.set_data(x, df["flare_kgph"].iloc[:i+1])
    ax.set_title(f"t={df['t_s'].iloc[i]:.0f}s | T={df['reactor_C'].iloc[i]:.0f}°C | Profit={df['profit_rsph'].iloc[i]:.0f} Rs/h")
    return line_oil, line_char, line_flare

ani = FuncAnimation(fig, update, frames=len(df), init_func=init, interval=80, blit=True)

gif_path = "digital_twin_fast.gif"
ani.save(gif_path, writer="pillow", fps=8)  # lower fps = faster
gif_path
